In [ ]:
import os
import lightning.pytorch as lightning
import torch
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import torch.nn as nn


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
Path to dataset files: /kaggle/input/chest-xray-pneumonia


In [ ]:
testData = mnist_dataset = datasets.MNIST(root='./data', train=False, transform=transforms.ToTensor(), download=True)
trainData = testData = mnist_dataset = datasets.MNIST(root='./data', train=False, transform=transforms.ToTensor(), download=True)

In [ ]:
# Extra params for training

learning_rate = 0.01
batch_size = 30
epochs = 15

In [ ]:
class PneumoniaModel(lightning.LightningModule):
    def __init__(self, num_classes, lr):
        super(PneumoniaModel, self).__init__()
        # Initialize the pretrained ResNet18 model
        self.model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        # Freeze all layers of the model
        for param in self.model.parameters():
            param.requires_grad = False

        # Modify the fully connected layer to match the number of classes
        features = self.model.fc.in_features
        self.model.fc = nn.Linear(features, num_classes)

        # Store the learning rate
        self.lr = lr

    def forward(self, x):
        # Define the forward pass (basically calling the model)
        return self.model(x)

    def training_step(self, batch, batch_idx):
        # Get inputs and labels from the batch
        images, labels = batch

        # Get predictions from the model
        logits = self(images)

        # Compute the loss (cross-entropy)
        loss = F.cross_entropy(logits, labels)

        # Log the loss for visualization (optional)
        self.log('train_loss', loss)

        return loss

    def validation_step(self, batch, batch_idx):
        # Similar process for validation
        images, labels = batch
        logits = self(images)
        loss = F.cross_entropy(logits, labels)

        # You can also log other metrics like accuracy here
        acc = (logits.argmax(dim=1) == labels).float().mean()
        self.log('val_loss', loss)
        self.log('val_accuracy', acc)

        return loss

    def configure_optimizers(self):
        # Only optimize the fully connected layer
        optimizer = torch.optim.Adam(self.model.fc.parameters(), lr=self.lr)
        return optimizer

# To initialize and train the model:
# model = PneumoniaModel(num_classes=2, lr=0.001)


In [ ]:
def get_dataloaders(data_dir="chest_xray", batch_size=32):
    imagenet_mean = [0.485, 0.456, 0.406]
    imagenet_std = [0.229, 0.224, 0.225]

    train_transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
    ])

    val_test_transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
    ])

    train_data = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=train_transform)
    val_data = datasets.ImageFolder(os.path.join(data_dir, "val"), transform=val_test_transform)
    test_data = datasets.ImageFolder(os.path.join(data_dir, "test"), transform=val_test_transform)

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader, train_data.classes

In [ ]:
# Early stopping + logs
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import EarlyStopping

def train():
  log = CSVLogger('lightning_logs', name='MLP')
  early_stopping = EarlyStopping(monitor='val_loss', patience=3, mode='min')
  trainer = lightning.Trainer(max_epochs=epochs, logger=log, callbacks=[early_stopping])
  model = PneumoniaModel(3, learning_rate)
  train, val, test, classes = get_dataloaders(path + "/chest_xray")
  trainer.fit(model, train, val)
  trainer.test(model, test)

In [ ]:
train()

INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 
  | Name  | Type   | Params | Mode 
-----------------------------------------
0 | model | ResNet | 11.2 M | train
-----------------------------------------
1.5 K     Trainable params
11.2 M    Non-trainable params
11.2 M    Total params
44.712    Total estimated model params size (MB)
68        Modules in train mode
0         Modules in eval mode
INFO:lightning.pytorch.callbacks.model_summary:
  | Name  | Type   | Params | Mode 
-----------------------------------------
0 | model | ResNet | 11.2 M | train
-----------------------------------------
1.5 K     Trainable params
11.2

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]